# Contrastive Miner - Hard Negative Mining Demo

This notebook demonstrates how to use the `contrastive-miner` library to mine hard negatives from your existing training data.

**Two-Stage Mining Approach:**
1. **Stage 1 (Document Retrieval)**: Retrieve similar chunks → re-rank → select hard negatives
2. **Stage 2 (Query Similarity)**: Find similar queries → get their positives → re-rank → select hard negatives

In [16]:
import json
import random

from sentence_transformers import SentenceTransformer
from contrastive_miner import (
    SemanticNegativeMiner,
    MinerConfig,
    group_by_anchor,
    load_data
)

## 1. Load Data

Load the existing training data and take a small sample for demonstration.

In [17]:
# Load the original dataset
DATA_PATH = "../../fine-tuning-embeddings-on-domain-specific-data/data/train.json"

with open(DATA_PATH, 'r') as f:
    full_data = json.load(f)

print(f"Total rows in dataset: {len(full_data):,}")
print(f"Sample row: {list(full_data[0].keys())}")

Total rows in dataset: 7,058
Sample row: ['query', 'positive']


In [18]:
# Take a small sample for demo (100 rows)
SAMPLE_SIZE = 1000

random.seed(42)
sample_data = random.sample(full_data, SAMPLE_SIZE)

print(f"Sample size: {len(sample_data)}")
print("\nFirst sample:")
print(f"  Query: {sample_data[0]['query'][:80]}...")
print(f"  Positive: {sample_data[0]['positive'][:80]}...")

Sample size: 1000

First sample:
  Query: What are the factors considered in determining if a company's goodwill is impair...
  Positive: Valua on of goodwill and long-lived assets.  We perform an annual impairment rev...


## 2. Group by Anchor

Since multiple queries may map to the same positive chunk, we group them to avoid false negatives.

In [19]:
# Group data by anchor (query) - this collects all positives per query
grouped_data = group_by_anchor(sample_data)

print(f"Grouped into {len(grouped_data)} unique anchors")
print("\nExample (with multiple positives if any):")
for item in grouped_data[:3]:
    print(f"  Anchor: {item['anchor'][:60]}...")
    print(f"  Positives: {len(item['positives'])} chunk(s)")
    print()

Grouped into 995 unique anchors

Example (with multiple positives if any):
  Anchor: What are the factors considered in determining if a company'...
  Positives: 1 chunk(s)

  Anchor: Who signed the Performance Stock Unit Award Agreement, dated...
  Positives: 1 chunk(s)

  Anchor: What information can be found under ITEM 7A?...
  Positives: 1 chunk(s)



## 3. Configure Mining

Set up the embedding model and mining configuration.

In [20]:
# Load embedding model (user provides this)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Loaded embedder: {embedder}")

Loaded embedder: SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [21]:
# Configure mining parameters
config = MinerConfig(
    # Stage 1: Document retrieval
    stage1_n_hard=2,        # 2 hard negatives from document retrieval
    stage1_n_random=1,      # 1 random negative
    stage1_retrieve_buffer=10,
    
    # Stage 2: Query similarity
    stage2_top_queries=10,  # Look at top 10 similar queries
    stage2_n_hard=2,        # 2 hard negatives from query similarity
    stage2_n_random=1,      # 1 random negative
    
    batch_size=32
)

print("Mining Configuration:")
print(config.model_dump_json(indent=2))

Mining Configuration:
{
  "stage1_n_hard": 2,
  "stage1_n_random": 1,
  "stage1_retrieve_buffer": 10,
  "stage2_top_queries": 10,
  "stage2_n_hard": 2,
  "stage2_n_random": 1,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "use_reranker": false,
  "reranker_model": null,
  "batch_size": 32
}


## 4. Create Miner and Mine Negatives

The miner will:
1. Build vector indices for chunks and queries
2. For each query, run both mining stages
3. Output intermediate (stage-specific) and final (flattened) formats

In [22]:
# Create miner (uses default cross-encoder for reranking)
miner = SemanticNegativeMiner(embedder, config)

# Alternatively, provide custom reranker:
# from sentence_transformers import CrossEncoder
# miner = SemanticNegativeMiner(embedder, config, cross_encoder=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2"))

In [23]:
# Mine negatives!
intermediate_rows, triplet_rows = miner.mine_dataset(
    grouped_data,
    intermediate_path="../data/intermediate_sample.jsonl",  # Stage-specific output
    final_path="../data/triplets_sample.jsonl"              # Final flattened output
)

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Mining: 100%|██████████| 995/995 [01:19<00:00, 12.55it/s]


## 5. Inspect Results

In [24]:
print(f"Generated {len(intermediate_rows)} intermediate rows")
print(f"Generated {len(triplet_rows)} triplet rows")

Generated 1000 intermediate rows
Generated 1000 triplet rows


In [25]:
# Inspect intermediate format (stage-specific negatives)
sample_intermediate = intermediate_rows[0]

print("=" * 80)
print("INTERMEDIATE FORMAT (Stage-Specific)")
print("=" * 80)
print(f"\nAnchor: {sample_intermediate.anchor[:80]}...")
print(f"\nPositive: {sample_intermediate.positive[:80]}...")
print(f"\nStage 1 - Hard Negatives (Doc Retrieval): {len(sample_intermediate.hard_neg_doc)}")
for i, neg in enumerate(sample_intermediate.hard_neg_doc):
    print(f"  [{i+1}] {neg[:60]}...")
print(f"\nStage 1 - Random Negatives: {len(sample_intermediate.random_neg_doc)}")
print(f"\nStage 2 - Hard Negatives (Query Similarity): {len(sample_intermediate.hard_neg_query)}")
for i, neg in enumerate(sample_intermediate.hard_neg_query):
    print(f"  [{i+1}] {neg[:60]}...")
print(f"\nStage 2 - Random Negatives: {len(sample_intermediate.random_neg_query)}")

INTERMEDIATE FORMAT (Stage-Specific)

Anchor: What are the factors considered in determining if a company's goodwill is impair...

Positive: Valua on of goodwill and long-lived assets.  We perform an annual impairment rev...

Stage 1 - Hard Negatives (Doc Retrieval): 2
  [1] Table of Contents
Goodwill.  Goodwill represents the excess ...
  [2] Table of Contents
Business Combinations
The Company is requi...

Stage 1 - Random Negatives: 1

Stage 2 - Hard Negatives (Query Similarity): 2
  [1] Table of Contents
Goodwill.  Goodwill represents the excess ...
  [2] Table of Contents
Business Combinations
The Company is requi...

Stage 2 - Random Negatives: 1


In [14]:
# Inspect final format (flattened for training)
sample_triplet = triplet_rows[0]

print("=" * 80)
print("FINAL FORMAT (Flattened for Training)")
print("=" * 80)
print(f"\nAnchor: {sample_triplet.anchor[:80]}...")
print(f"\nPositive: {sample_triplet.positive[:80]}...")
print(f"\nNegatives ({len(sample_triplet.negatives)} total):")
for i, neg in enumerate(sample_triplet.negatives[:5]):
    print(f"  [{i+1}] {neg[:60]}...")

FINAL FORMAT (Flattened for Training)

Anchor: What are the factors considered in determining if a company's goodwill is impair...

Positive: Valua on of goodwill and long-lived assets.  We perform an annual impairment rev...

Negatives (5 total):
  [1] Table of Contents Alphabet Inc.
• litigation or other claims...
  [2] Table of Contents
Critical Accounting Estimates
Our discussi...
  [3] 9 
  
STOCK PERFORMANCE  
COMPARISON OF 5 YEAR CUMULATIVE TO...
  [4] Table of Contents
contracts with our materials providers and...
  [5] (the “PLOA Policy”) and the leave exceeds a certain duration...


## 6. Saved Files

The miner saved two files:
- **Intermediate**: `data/intermediate_sample.jsonl` - Stage-specific negatives for debugging/analysis
- **Final**: `data/triplets_sample.jsonl` - Flattened format ready for Triplet Loss training

In [26]:
# Load and verify saved files
import os

print("Saved files:")
for fname in ["intermediate_sample.jsonl", "triplets_sample.jsonl"]:
    path = f"../data/{fname}"
    if os.path.exists(path):
        with open(path) as f:
            lines = sum(1 for _ in f)
        size = os.path.getsize(path) / 1024
        print(f"  {fname}: {lines} rows, {size:.1f} KB")

Saved files:
  intermediate_sample.jsonl: 1000 rows, 21826.0 KB
  triplets_sample.jsonl: 1000 rows, 19790.0 KB
